# Structured Output: Getting JSON from an LLM 🧾

So far our LLM hands back **prose** — paragraphs meant for humans to read. That's great for a chatbot, but useless the moment your *code* needs to actually use the answer.

> If you're building an app, you rarely want a paragraph. You want **data**: a price as a number, ingredients as a list, a rating you can store in a database.

This lesson is about forcing the LLM to reply in a clean, predictable shape — **JSON** — that Python can read directly. This is the skill that turns "a cool demo" into "a real app." Builds on lesson 4 (LLM calls) and the classes lesson (you'll see why in a minute).

## 1. The problem: prose is useless to code

Let's ask for a recipe the old way and then try to *use* one specific fact from it — say, the calories.

In [1]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

In [2]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Give me a simple Indian dal recipe (in less than 10 lines) with its calorie count."
)
print(response.text)

**Ingredients:** 1/2 cup yellow moong dal, 2 cups water, 1/2 tsp turmeric, salt, 1 tbsp ghee, 1 tsp cumin seeds, 1 green chili (chopped).  
**Step 1:** Rinse the dal and boil it with water, turmeric, and salt until completely soft (about 15 minutes).  
**Step 2:** Mash the dal lightly with a spoon to achieve a smooth consistency.  
**Step 3:** In a small pan, heat ghee, add cumin seeds and green chili, and sauté for 30 seconds until fragrant.  
**Step 4:** Pour the hot tempering over the boiled dal, stir well, and serve hot.  
**Calories:** ~175 kcal per serving (Recipe makes 2 servings).


Now imagine your app needs *just the calorie number* to show on a dashboard. How do you pull it out of that paragraph? You can't reliably — it might say "around 250 calories", "250-300 kcal", or bury it in sentence three.

Trying to extract a number from free-form text is like asking a friend for directions and getting a 10-minute story about their cousin's wedding. 🙃 Somewhere in there is the turn you needed... probably.

## 2. First attempt: just *ask* for JSON

**JSON** is simply text in a strict key–value format that every programming language understands. Let's ask the model to use it — and watch a classic gotcha bite us.

In [4]:
import json

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents='''Give me a simple Indian dal recipe.
    Return ONLY JSON with these keys: title, calories, prep_time_minutes.'''
)
print(response.text)  #👀 look closely at how it starts and ends...

{
  "title": "Simple Yellow Moong Dal",
  "calories": 180,
  "prep_time_minutes": 10
}


In [5]:
#Looks like json, so let's just parse it
data = json.loads(response.text)
print(data)

{'title': 'Simple Yellow Moong Dal', 'calories': 180, 'prep_time_minutes': 10}


## 3. The robust way: tell Gemini the *exact* shape you want

Instead of begging in the prompt, we hand Gemini a **schema** — a precise description of the fields and their types. The model is then *forced* to fill in exactly that shape. No fences, no chit-chat, no surprises.

We describe the shape with a **Pydantic model** — the same data-model class you met in the **classes** lesson. Quick recap: list the fields with their types, and Pydantic enforces them.

In [17]:
from pydantic import BaseModel
from google.genai import types

# This class IS the shape we want back. Note the types: str, int, list of str.
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    calories: int
    prep_time_minutes: int

response = client.models.generate_content(
    model = "gemini-3.8-flash",
    contents = "Give me a simple Indian dal recipe.",
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",    #Reply in JSON
        response_schema = Recipe,                   #Exaclty in this format
    ),
)
print(response.text)

{"title":"Simple Tadka Dal","ingredients":["1 cup yellow split peas (toor dal or moong dal)","3 cups water","1/2 tsp turmeric powder","1 tsp salt","1 tbsp ghee or vegetable oil","1 tsp cumin seeds","2 cloves garlic, minced","1 small onion, finely chopped","1 green chili, slit","Fresh cilantro, chopped"],"calories":210,"prep_time_minutes":30}


And the best part — because we gave it a schema, the library can hand us back a ready-made `Recipe` **object** (no `json.loads` needed). It's real Python data now:

In [23]:
recipe = response.parsed   # a Recipe object, parsed for us

print("Title:", recipe.title)
print("Calories:", recipe.calories)     # a real int
print("Prep_time:", recipe.prep_time_minutes, "min")

print("Ingredients:")
for item in recipe.ingredients:
    print(" -", item)

Title: Simple Tadka Dal
Calories: 210
Prep_time: 30 min
Ingredients:
 - 1 cup yellow split peas (toor dal or moong dal)
 - 3 cups water
 - 1/2 tsp turmeric powder
 - 1 tsp salt
 - 1 tbsp ghee or vegetable oil
 - 1 tsp cumin seeds
 - 2 cloves garlic, minced
 - 1 small onion, finely chopped
 - 1 green chili, slit
 - Fresh cilantro, chopped


🎉 That's the whole game. The LLM's answer is now structured data you can drop into a database, a Streamlit table, an API response, or a calculation. No guessing, no fragile text-parsing.

**Rule of thumb:** chatting with a human → plain text. Feeding the result into code → structured output with a schema.